<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset.

We import the Yelp dataset from Kaggle, using a token.

In [51]:
import os
import json
import pandas as pd
import pip
import string

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"
!kaggle datasets download -d yelp-dataset/yelp-dataset

yelp-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


We then prepare the entry point for the Spark functionalities that will we use from now on.

In [52]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

In [53]:
import zipfile
from multiprocessing import Pool

DATA_DIR = "/content/yelp-dataset"
FILE_NAME = "yelp_academic_dataset_review.json"
with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
    zip_ref.extract(member = FILE_NAME, path=DATA_DIR)

We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`. We convert the resulting dataframe in RDD form, seen that from now on we will manage data in this format.

In [54]:
all_reviews_RDD = sc.textFile(DATA_DIR + "/" +FILE_NAME).map(lambda s: json.loads(s))

Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews. From now on we will refer to the `text` field of a review simply as *review*.

In [55]:
all_reviews_RDD = all_reviews_RDD.map((lambda r: (r['review_id'], r['text']))).cache()

We sample the dataset at the only scope to speed up the following steps in the (free) Google Colab environment. It is possible to adjust the number of reviews by setting the variable `sample_size`.

Note that to sample the rows from `all_reviews_RDD` we collect them in local memory since that we are interesed in a small portion of data. For a more sophisticated, but heavier, approach refer to the `sample` method, as shown in the commented line, where `f` is the fraction of rows to be chosen from the RDD to sample.

In [56]:
# reviews_RDD = all_reviews_RDD.sample(withReplacement = False, fraction = f, seed = 42)

sample_size=100
reviews_RDD = sc.parallelize(all_reviews_RDD.take(sample_size))
review_num = reviews_RDD.count()
print("The chosen random sample contains {} reviews".format(review_num))

The chosen random sample contains 100 reviews


To have a better idea, let's look at a review in the dataset.

In [57]:
ex_review,text = reviews_RDD.first()
print('review key: {}\n ---\n{}'.format(ex_review,text))

review key: KU_O5udG6zpxOg-VcAEodg
 ---
If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. 

The food is good, but it takes a very long time to come out. The waitstaff is very young, but usually pleasant. We have just had too many experiences where we spent way too long waiting. We usually opt for another diner or restaurant on the weekends, in order to be done quicker.


## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that each review is nonempty.

In [58]:
reviews_RDD = reviews_RDD.filter(lambda text: bool(text))

To study the similarity between reviews we look at the corresponding strings as sets of substrings. We take into consideration two approches to divide the text into items, described in the sections below. In both cases we maitain each item apprearing in the whole dataset once.    

### Shingles

We look at the reviews from more of a sintactical point of view by concentrating on the construction of the sentence. In this instance we look at all the substrings of length $k$ contained in the strings, called $k$-grams or *shingles*.
The value of $k$ is chosen on the basis of the length of the document to be subdivided. We observe that the number of possible $k$-grams is in the order of the quantity of characters that compose it (to be exact $l-k+1$ where $l$ is the document length). So we would like to have $k$ great enough such that the probability that of finding a specific shingle in a document is small. It also to be said that it is very unlikely that all shingles appear in a corpus of text. To take this into account one may consider a larger $k$ as to diffirentiate the documents better. Since we will not encounter all of those shingles, we can hash those elements onto a bucket that contains only all possible $k$-grams. Here we choose to hash $9$-grams onto possible $5$-grams, and we leave the hashed singles in numeric form.

In [59]:
import hashlib
import math

def get_shingles(text:string, k:int):
    """ Extract k-gram

        Args:
            text (string): string from which to extract the shingles
            k (int): length of the shingles

        Returns:
            list: list contaning all k-grams in text (all substrings of length k)
        """
    k_grams = []
    for i in range(len(text) - k + 1):
        k_grams.append(text[i:i+k])
    return k_grams

def RDD_unique_shingles(rdd,k:int):
    """ Extract all unique k-grams from all texts

        Args:
            rdd (RDD): RDD of (key,text)
            k (int): size of the k-grams

        Returns:
            RDD: RDD of (key,list of shingles extracted from text)
        """
    return (rdd
            .map(lambda s: (s[0], get_shingles(s[1],k)))
            .map(lambda s: (s[0], list(set(s[1])))))

def hash_object(byte_obj:bytes, seed:int, hash_bucket_size:int):
    """ Hash an object into a bucket of values [0,hash_buckt_size-1] on the basis of the seed

        Args:
            byte_obj (byte array): an object in byte format
            seed (int): seed for the hash function
            hash_bucket_size (int): the size of the bucket to which the objects get hashed to

        Returns:
            int: hashed object
        """
    m=hashlib.shake_256()
    m.update(byte_obj)
    m.update(bytes(seed))
    hash_bucket_size = math.floor(hash_bucket_size/256)
    return int.from_bytes(m.digest(hash_bucket_size),'little')

def RDD_unique_hashed_shingles(rdd,larger_k:int,k:int):
    """ Hash all shingles of all texts

        Args:
            rdd (RDD): RDD of (key, text)
            larger_k (int): length of the original shingles
            k (int): final shingle size - dnumber of bytes that the hashed shingles may occupy

        Returns:
            RDD: RDD of (key, list of hashed shingles)
        """
    assert(larger_k >= k)
    return RDD_unique_shingles(rdd,larger_k).map(lambda s: (s[0],[hash_object(bytearray(g,'utf-8'),42,5*256) for g in s[1]]))

Let's glance at how the last of the reviews we shown above has been transformed into shingles. To be noted that the order of the shingles does not follow the actual sentence.

In [60]:
larger_k=9
k=5

shingles_in_review_RDD = RDD_unique_shingles(reviews_RDD,larger_k)
hashed_shingles_in_review_RDD = RDD_unique_hashed_shingles(reviews_RDD,larger_k,k).cache()

In [61]:
hashed_shingles_in_review_RDD.first()

('KU_O5udG6zpxOg-VcAEodg',
 [367661289858,
  1044435119417,
  903794704600,
  620481552778,
  263110748580,
  673352737248,
  86055717547,
  797021438592,
  473973936863,
  116972837731,
  205864672315,
  986727378030,
  46568886886,
  75319143547,
  738506472184,
  759868362014,
  698615024882,
  475682759600,
  16863190468,
  863044212626,
  526513531917,
  531128663624,
  87722525112,
  513373836701,
  250707211053,
  456888366057,
  798470716883,
  307421913598,
  232531810433,
  407442263322,
  611094389759,
  723051288604,
  1009193315349,
  670466626889,
  146007623237,
  555031556481,
  60088263165,
  813204450625,
  717621948230,
  102582788641,
  87846454337,
  753025765807,
  809090230776,
  126374478933,
  306181533297,
  766794755629,
  875079518208,
  917743772649,
  78565618407,
  91764414600,
  408938517731,
  713159346968,
  823105761317,
  792906334200,
  234965640655,
  267538305928,
  146698400769,
  137844280598,
  757925623160,
  363976392501,
  660412571296,
  38

In [62]:
_, shingles = shingles_in_review_RDD.filter(lambda s: s[0] == ex_review).first()
hashed_shingles = hashed_shingles_in_review_RDD.filter(lambda s: s[0] == ex_review).collect()[0][1]
print("review key:\t\t{}\n {}-grams:\t\t{}\n corresponding {}-grams: {}".format(ex_review,larger_k,shingles,k,hashed_shingles))

review key:		KU_O5udG6zpxOg-VcAEodg
 9-grams:		['are it is', 'ry long t', 'had a bad', ' too many', 'riences w', 'o long wa', 'ke about ', 'ed it mul', 'ide to ea', 'd, but it', 'er to be ', 'e weekend', 's very yo', 'ere we sp', 'm beginni', ' waiting.', 'de to eat', 'I want to', ' takes a ', ' I want t', ' it multi', 'e. \n\nThe ', 'be done q', 'o take ab', 'ng, but u', 'r to be d', ' be aware', 'it is goi', 'e we spen', 'eekends, ', 'f is very', ' done qui', 'decide to', 'but usual', 'ually ple', 'to be don', ' it takes', 'ning to e', ' is good,', 'ad experi', 'tstaff is', '. The wai', 'asant. We', 'u decide ', 't here, j', 'tiple tim', 've just h', 'ut. The w', 'rs from b', 'be aware ', 'rder to b', 'end. We h', 'ther loca', 'e tried i', ' pleasant', ' is very ', '. We usua', 'ood is go', ' from beg', 'waitstaff', 'ant to li', 'ried it m', 'aff is ve', 'ons in NJ', 'ave just ', ' come out', 'ng time t', 'ant. We h', 'to take a', 've tried ', 'ut 2 hour', 'd too man', 'out. The ', '

### Tokens

We give more importance to the sematical point of view, by looking at the lemmas that form a sentence. In this case we divide the reviews in the terms that compose it, and to emphasize the meaning of the review we have modified the reviews as follows:
* we get rid of the stop words appearing in the tokens to extract the actual semantics of the text.
* we lemmatize the remaining tokens: we are oblivious about the various inflections of a certain word.

We then form shingles made of $k$ of such lemmas, from here on called shingles *tokens*.

In [63]:
%%capture
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('english')
import spacy
nlp = spacy.load("en_core_web_sm")

split_regex = r'\W+'
get_tokens = lambda string: [s for s in re.split(split_regex,string.lower()) if s not in stopwords and s != '']
lemmatize = lambda word: str([token.lemma_ for token in nlp(word)][0])

def RDD_unique_lemmatized_tokens(rdd):
    """ For each key find the unique lemmas in the text

        Args:
            rdd (RDD): RDD of (key, text)

        Returns:
            RDD: RDD of (key, list of unique lemmas in text)
        """
    return (rdd
            .map(lambda s: (s[0], get_tokens(s[1])))
            .map(lambda s: (s[0], [lemmatize(w) for w in s[1]]))
            .map(lambda s: (s[0], list(set(s[1])))))

We inspect how the last of the reviews shown above has been transformed into tokens.

In [64]:
tokens_in_review_RDD = RDD_unique_lemmatized_tokens(reviews_RDD).map(lambda s: (s[0],get_shingles(s[1],3)))

_, tokens = tokens_in_review_RDD.filter(lambda s: s[0] == ex_review).first()
print("review key:\t {}\n tokens:\t {}".format(ex_review, tokens))

review key:	 KU_O5udG6zpxOg-VcAEodg
 tokens:	 [['2', 'wait', 'eat'], ['wait', 'eat', 'decide'], ['eat', 'decide', 'want'], ['decide', 'want', 'food'], ['want', 'food', 'try'], ['food', 'try', 'location'], ['try', 'location', 'another'], ['location', 'another', 'hour'], ['another', 'hour', 'bad'], ['hour', 'bad', 'nj'], ['bad', 'nj', 'multiple'], ['nj', 'multiple', 'usually'], ['multiple', 'usually', 'way'], ['usually', 'way', 'do'], ['way', 'do', 'go'], ['do', 'go', 'come'], ['go', 'come', 'pleasant'], ['come', 'pleasant', 'spend'], ['pleasant', 'spend', 'like'], ['spend', 'like', 'good'], ['like', 'good', 'waitstaff'], ['good', 'waitstaff', 'order'], ['waitstaff', 'order', 'take'], ['order', 'take', 'aware'], ['take', 'aware', 'young'], ['aware', 'young', 'time'], ['young', 'time', 'never'], ['time', 'never', 'weekend'], ['never', 'weekend', 'diner'], ['weekend', 'diner', 'begin'], ['diner', 'begin', 'experience'], ['begin', 'experience', 'opt'], ['experience', 'opt', 'quicker'], ['op

At this point we have codified each review through it's essential information.

## Similar items with Jaccard similarity

In this section we use the `shingles_in_review_RDD` RDD containing the shingles for a review as an example to help illustrate the behaviour of the described methods.

We begin by defining the similarity measure we wil be using to compare reviews. Jaccard similarity between sets $S$ and $T$ is defined as:
\begin{equation*}
J(S,T)=\frac{|S\cap T|}{|S\cup T|}
\end{equation*}
In our case the items of the sets are the shingles that we have extracted from the reviews, so the sets are the summaries of the reviews.

In [65]:
def Jaccard(S:set,T:set):
    """ Compute Jaccard similarity between two sets

        Args:
            S (set): collection of homogeneous objects
            T (set): collection of homogeneous objects

        Returns:
            int: Jaccard similarity between s and t
        """
    return len(S.intersection(T)) / len(S.union(T))

### Similarity preserving summary of the reviews
To store the shingles extracted from each review compactly we consider their characteristic matrix, used to represent a collection of sets. Such a structure is defined as a binary matrix, which holds all the reviews in the columns, and all possible shingles in the rows. A cell is set to $1$ only if the corresponding token appears in the indicated review, and $0$ otherwise.

Though with growing amounts of data it is not realistic to maintain the whole characteristic matrix. So, we construct a succint structure called a signature matrix that scales down the number of rows.    
Each row of such matrix is constructed as follows:
1. choose a permutation of the rows of the characteristic matrix uniformly at random among all possible permutations,
2. apply the chosen permutation to the rows of the charactistic matrix,
3. apply a minhash function to all columns of the resulting matrix.

A minhash function is defined as $h:\{\text{reviews}\}\to\{\text{shingles}\}$, and for a review it returns the index of the first token appearing in the column of the characteristic matrix.

The column related to a review in the signature matrix is it's signature. This signature depends on the random permutations applied to the rows of the chacteristic matrix.

The length of the signatures can be adjusted by setting the variable `signature_length`.

In [66]:
signature_length = 100

We prepare two functions that we will use in the next blocks to transpose matrices represented with RDDs. In case of sparse characteristic matrices we considered it easier to look at the list of items that form a certain set.

In [67]:
def RDD_transpose_sparse_characteristic(items_in_row_rdd):
    """ Transpose a sparse characteristic matrix

        Args:
            items_in_row_rdd (RDD): RDD of (row key, list of items in row)

        Returns:
            RDD: RDD of (item, row keys contained in item column)
        """
    return (items_in_row_rdd
            .flatMap(lambda s: [(t,s[0]) for t in s[1]])
            .groupByKey().mapValues(list))

def RDD_transpose(items_in_row_rdd, columns_rdd):
    """ Transpose a matrix

        Args:
            items_in_row_rdd (RDD): RDD of (row key, list of items in row)
            columns_rdd (RDD): RDD of (column name, -)

        Returns:
            RDD: RDD of (item, row keys contained in item column)
        """
    item_pos_in_row_RDD = items_in_row_rdd.map(lambda s: (s[0], [(s[1].index(n),n) for n in s[1]])).cache()
    item_pos_perm_value_RDD = (item_pos_in_row_RDD
                                .flatMap(lambda s: [(t[0],(s[0],t[1])) for t in s[1]])
                                .groupByKey().mapValues(list)
                                .map(lambda s: (s[0],[t[1] for t in sorted(s[1])]))).cache()
    columns_indexed_RDD = columns_rdd.map(lambda s: s[0]).zipWithIndex().map(lambda s: (s[1],s[0])).cache()
    return item_pos_perm_value_RDD.join(columns_indexed_RDD).map(lambda s: (s[1][1],s[1][0]))

To build the signature matrix we start by fixing all the permutations of the rows of the characteristic matrix, in number equal to the length of the signatures.

In [68]:
import numpy as np

def RDD_permutation_positions(n, permutations_num):
    """ For each index in [0,permutations_num-1] return a permutation on the values in {0,1,...,n-1}

        Args:
            n (int): number of values to permute
            permutations_num (int): number of permutations to produce

        Returns:
            RDD: RDD of (permutation index, list of values that form a permutation of {0,1,...,n-1})
        """
    permutations_num_bc = sc.broadcast(permutations_num)
    n_bc = sc.broadcast(n)
    return sc.parallelize(range(permutations_num_bc.value)).map(lambda s: (s,list(np.random.permutation(n_bc.value))))

At this point we determine for all possible shingles the position that it assumes for each of the fixed permutations.

In [69]:
def RDD_item_indices_for_set(items_in_set_rdd, permutations_num):
    """ For each set specify the positions that the relative items assume for each permutation

        Args:
            items_in_set_rdd (RDD): RDD of (set key, list of items contained in set)
            permutations_num (int): permutations_num (int): numbers of permutations to produce for the items

        Returns:
            RDD: RDD ((set key, permutation index), list of positions assumed by the items in set)
        """
    item_RDD = RDD_distinct_items(items_in_set_rdd)
    items_num = item_RDD.count()
    indexed_item_RDD = item_RDD.map(lambda s: s[0]).zipWithIndex().cache()
    sets_for_item_RDD = RDD_transpose_sparse_characteristic(items_in_set_rdd).cache()
    sets_for_indexed_items_RDD = sets_for_item_RDD.join(indexed_item_RDD).map(lambda s: ((s[0],s[1][1]),s[1][0])).cache()
    indexed_items_in_set_RDD = RDD_transpose_sparse_characteristic(sets_for_indexed_items_RDD).cache()
    items_perm_pos_RDD = RDD_permutation_positions(items_num, permutations_num).cache()
    return (indexed_items_in_set_RDD
            .cartesian(items_perm_pos_RDD)
            .map(lambda s:((s[1][0], s[0][0]),[(t[0],s[1][1][t[1]]) for t in s[0][1]])))

From these couples it is immediate to compute the minhash for each review for all the fixed permutations. Specifically, we have found the signature for each review.

In [70]:
def RDD_distinct_items(items_in_set_rdd):
    """ Collect all distinct appearing items

        Args:
            items_in_set_rdd (RDD): RDD of (set key, list of items contained in set)

        Returns:
            RDD: RDD of (item, number of occurences of item in items_in_set_rdd)

        """
    return (items_in_set_rdd
            .flatMap(lambda t: [(s, 1) for s in t[1]])
            .reduceByKey(lambda a,b: a+b))

def RDD_minhashes(items_in_set_rdd, permutations_num):
    """ For each set compute the relative minhash

        Args:
            items_in_set_rdd (RDD): RDD of (set key, list of items contained in set)
            permutations_num (int): number of permutations according to which to compute the minhashes

        Returns:
            RDD: RDD of (item, number of occurences of item in items_in_set_rdd)
        """
    return (RDD_item_indices_for_set(items_in_set_rdd, permutations_num)
            .map(lambda s: (s[0],min([t[1] for t in s[1]]))))

def RDD_signatures(items_in_set_rdd, signature_length):
    """ For each set compute the signature

        Args:
            items_in_set_rdd (RDD): RDD of (set key, list of items contained in set)
            signature_length (int): desired lenght of the signatures for the sets

        Returns:
            RDD: RDD of (item, number of occurences of item in items_in_set_rdd)
        """
    return (RDD_minhashes(items_in_set_rdd, signature_length)
            .map(lambda s: (s[0][1],s[1])).groupByKey().mapValues(list))

Let's give a look at the signature for a given review.

In [71]:
review_signature_RDD = RDD_signatures(hashed_shingles_in_review_RDD, signature_length).cache()

review, signature = review_signature_RDD.first()
print("review key:\t {}\n signature:\t {}".format(review,signature))

review key:	 6AxgBCNX_PNTOxmbRSwcKQ
 signature:	 [8, 34, 77, 54, 3, 17, 59, 58, 141, 154, 77, 14, 108, 9, 31, 21, 112, 28, 60, 54, 78, 65, 38, 47, 38, 37, 82, 9, 11, 45, 9, 4, 8, 5, 1, 25, 21, 53, 16, 0, 23, 26, 10, 71, 74, 106, 54, 204, 14, 121, 8, 34, 77, 54, 3, 17, 59, 58, 141, 154, 77, 14, 108, 9, 31, 21, 112, 28, 60, 54, 78, 65, 38, 47, 38, 37, 82, 9, 11, 45, 9, 4, 8, 5, 1, 25, 21, 53, 16, 0, 23, 26, 10, 71, 74, 106, 54, 204, 14, 121]


### Locality-sensitive hashing (LSH) & bounded Jaccard similarity

It would be unthinkable to compare all possible pairs of reviews to find similar ones among them. This would mean scanning all the rows in the signature matrix to compute the relative frequency between possible pairs of reviews. So, we proceed by applying Locality-Sensitive Hashing. In this approach we reduce the number of rows that determine the signature of a review by hashing so called bands of rows. The rationale is that similar reviews are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity. Looking at the resulting signatures we consider as a candidate pair only those for which their Jaccard similarity exceeds a threshold $t$.

We begin by dividing the signature matrix into $b$ bands of $r$ rows each. The choice of $r$ and $b$ depends on the threshold $t$ on the Jaccard similarity between pairs of reviews. The value of the threshold $t$ is approximately the value of similarity at which the probability of becoming a candidate is $\frac{1}{2}$.
So, we keep into account the following relationships (for brevity $l$=`signature_length`):   

\begin{equation*}
    \begin{cases}
        t=\left(\frac{1}{b}\right)^\frac{1}{r} \\
        r\cdot b = l
    \end{cases}
\end{equation*}

The function `band_size` solves this system in $r$ by computing it's value through
\begin{equation*}
    r=-\frac{W(-l\ln(t))}{\ln(t)}
\end{equation*}

where $W$ is the Lambert $W$ function, used to solve equations in the form $we^{w}=z$ for $w$.

In [72]:
from scipy.special import lambertw

def band_size(t, sig_len):
    """ Compute the number of rows that form a band for the LSH technique

    Args:
        t (int): desired threshold in [0,1] on the Jaccard similarity between pairs of sets
        sig_len (int): signature length for the sets

    Returns:
        int: ideal number of rows contained in the band
    """
    return -math.ceil(lambertw(-sig_len*np.log(t)).real/np.log(t))

def r_b_choice(t,sig_len):
    """ Choose the adeguate number of rows a band and the number of bands for the LSH technique

    Args:
        t (int): desired threshold in [0,1] on the Jaccard similarity between pairs of sets
        sig_len (int): signature length for the sets

    Returns:
        int: ideal number of rows, and consequent number of bands on the basis of the signature length
    """
    r=band_size(t,signature_length)
    b=math.ceil(signature_length/r)
    return (r,b)

In [73]:
t=0.05
t_bc = sc.broadcast(t)
r,b = r_b_choice(t,signature_length)
print("The chosen parameters are: \n r: {} \n b: {}".format(r,b))

The chosen parameters are: 
 r: 1 
 b: 100


Having chosen the parameters we proceed by subdividing the rows of the similarity matrix. In `review_banded_signature_RDD` this is simply done by storing for each review a sequence of lists, each of which contains the portion of the relative signature assigned to a band.

In [74]:
def split_list(l,b):
    """ Split list into b lists of equal length

    Args:
        l (list): list of elements
        b (int): number of sublists

    Returns:
        list: list formed by b sublists all of the same length, except for the last one if len(l) is not a multiple of b
    """
    split_list = [(list(a)) for a in np.array_split(np.array(l), b)]
    return [split_list[i] for i in range(b)]

We proceed by hashing for each token the rows in each of the $b$ bands. For each band we use a different hashing function, to avoid that the similarity between signatures depends on a specific hashing function. Subsequently signatures with two equal vectors in separate bands are hashed differently. Also, to avoid hashing distinct portions of a signature in the same bucket it is important to choose a great enough bucket.

In [75]:
def RDD_LSH(signatures_rdd, b, hash_bucket_size):
    """ Compute the hashed signatures with the LSH technique

    Args:
        signatures_rdd (RDD): RDD of (set key, signature for set)
        b (int): number of bands in which to split the signature matrix represented by signatures_rdd
        hash_bucket_size (int): the size of the bucket to which the signature portions in each band get hashed to

    Returns:

    """
    b_bc = sc.broadcast(b)
    hash_bucket_size_bc = sc.broadcast(hash_bucket_size)
    return (signatures_rdd
            .map(lambda s: (s[0],split_list(s[1],b_bc.value)))
            .map(lambda s: (s[0],[hash_object(bytes(str(t),'ascii'),i,hash_bucket_size_bc.value) for i,t in enumerate(s[1])])))

Let's give a look at the new compact representation of a review contained in `review_banded_signature`.

In [76]:
hash_bucket_size = 256*2
review_hashed_signature_RDD = RDD_LSH(review_signature_RDD, b, hash_bucket_size).cache()

review,signature = review_hashed_signature_RDD.first()
print("review key:\t {}\n signature:\t {}".format(review,signature))

review key:	 6AxgBCNX_PNTOxmbRSwcKQ
 signature:	 [56915, 57110, 26589, 53746, 23238, 12211, 37620, 9792, 57826, 49876, 5434, 2092, 26492, 65130, 13558, 13324, 22258, 29852, 23546, 53066, 5251, 17378, 22296, 63479, 20290, 12737, 43353, 64803, 35724, 52054, 25109, 3101, 33912, 38203, 22835, 19230, 18559, 20322, 53491, 59372, 33878, 32632, 24467, 13291, 8084, 7768, 28205, 39780, 24868, 49899, 54037, 48955, 5474, 42505, 49228, 314, 13228, 40609, 19873, 2973, 22980, 55316, 25993, 48035, 16881, 15518, 7088, 2613, 31146, 32289, 49712, 57714, 15752, 12268, 7233, 49992, 21283, 57141, 48063, 50526, 34165, 21112, 42836, 31355, 52770, 61754, 15894, 42484, 15754, 19320, 13183, 27286, 39778, 12096, 31013, 56669, 8752, 52153, 36044, 55957]


Now we search for candidate pairs among the reviews. We consider as possible similar couples of reviews those that have Jaccard similarity at least $t$.

In [77]:
def RDD_pairs(rdd):
    """ Put together all possible pairs of rows, without repetitions

    Args:
        rdd (RDD): RDD of (row key, information regarding row)

    Returns:
        RDD: RDD of ((r1,r2),(info1, info2)), with r1>r2 to avoid having duplicates
    """
    pairs_RDD = rdd.cartesian(rdd).filter(lambda s: s[0][0] > s[1][0])
    return pairs_RDD.map(lambda s: ((s[0][0], s[1][0]),(s[0][1], s[1][1])))

def RDD_similar_sets(rdd,t):
    """ Filter from pairs of rows whether they have Jaccard similarity at least t

    Args:
        rdd (RDD): RDD of (row key, information regarding the row)
        t (int): desired threshold on the Jaccard similarity

    Returns:
        RDD: RDD of ((r1,r2),1) if J(info1,info2)>=t
    """
    t_bc = sc.broadcast(t)
    pairs_RDD = RDD_pairs(rdd)
    return (pairs_RDD
            .filter(lambda s: Jaccard(set(s[1][0]),set(s[1][1]))>=t_bc.value)
            .map(lambda s: (s[0],1)))

The potential candidate pairs are the following.

In [78]:
candidate_pairs_LSH_RDD = RDD_similar_sets(review_hashed_signature_RDD,t).cache()
candidate_pairs_LSH_RDD.collect()

[]

For all of the candidates we verify their actual similarity, and we keep the reviews for which the Jaccard similarity computed on the respective signatures exceeds the threshold. We settle of the approximation of the Jaccard similarity computed though the relative frequency of equal elements in the signatures.

In [79]:
def filter_candidate_pairs(candidate_pair_rdd,set_exact_signature_rdd,t):
    """Given a set of candidate pairs of similar items, return those that are actually similar according to the exact signatures

    Args:
        candidate_pair_rdd (RDD): RDD of ((r1,r2),1) candidate pairs of similar items
        set_exact_signature_rdd (RDD): RDD of (set key, signature for set)
        t (int): desired threshold on the Jaccard similarity

    Returns:
        RDD: RDD of ((r1,r2),1) where J(exact_info1, exact_info2)>=t
    """
    candidate_pair_exact_signature_RDD = (candidate_pair_rdd
                                        .map(lambda s: (s[0][0],s[0][1]))
                                        .join(set_exact_signature_rdd)
                                        .map(lambda s: (s[1][0],(s[0],s[1][1])))
                                        .join(set_exact_signature_rdd)
                                        .map(lambda s: ((s[1][0][0],s[0]),(s[1][1],s[1][0][1]))))
    return RDD_similar_sets(candidate_pair_exact_signature_RDD,t)

Now we look at the actual similar pairs fo reviews in the dataset.

In [ ]:
similar_pairs_RDD = filter_candidate_pairs(candidate_pairs_LSH_RDD,review_signature_RDD,t)
similar_pairs_RDD.collect()

## Experiments
Like we have done for the `shingles_in_review_RDD`, we repeat the same procedure with different parameters. We will be varying the listed aspects:
* text subdivided in tokens or shingles
* $t$ the threshold on the Jaccard similarity
* the length of the signatures

In [ ]:
def run_experiment(items_in_set_rdd,t,signature_length,hash_bucket_size = 2*256):
    """ Run the process of finding similar items with threshold t on the Jaccard similarity with LSH technique

    Args:
        items_in_set_rdd (RDD): RDD of (set key, list of items contained in set)
        t (int): desired threshold in [0,1] on the Jaccard similarity between pairs of sets
        signature_length (int): desired lenght of the signatures for the sets
        hash_bucket_size (int): the size of the bucket to which the objects get hashed to

    Returns:

    """
    review_signature_RDD = RDD_signatures(rdd,signature_length)
    r,b = r_b_choice(t,signature_length)
    review_hashed_signature_RDD = RDD_LSH(review_signature_RDD,b,hash_bucket_size)
    RDD_similar_sets()

### Tokens - $t$=0.8


In [ ]:
RDD_signatures(tokens_in_review_RDD)

## Similar items evaluation

In [ ]:
from difflib import SequenceMatcher
s1="Definitely my favorite south city dive.   I probably wouldn't go drinking with the uptight people who gave this place a bad review.  But the people I've met there are my kind of bar crowd.  You go to a bar cuz there's a decent jukebox, cheap drinks, lots of things to drunkenly gamble on, fried food and you wanna shower when you get home.  5 checks."

s2="Reminds me of the backwards country bars I grew up sneaking in to.  Asked for a booth and was told the large booths were for parties of 12 or greater and the smaller booths were all full so we had to wait.  Waited a few minutes and the place was still dead, asked of we could get a larger booth and was told no.  I immediately left, completely rude and inconsiderate.  I have had my fantasy football draft here before and we ran up $600 tabs, never again."

s = SequenceMatcher(None, s1, s2)
s.ratio()

In [ ]:
import spacy

nlp = spacy.load("en_core_web_md")  # make sure to use larger package!
doc1 = nlp("I like salty fries and hamburgers.")
doc2 = nlp("Fast food tastes very good.")

# Similarity of two documents
print(doc1, "<->", doc2, doc1.similarity(doc2))
# Similarity of tokens and spans
french_fries = doc1[2:4]
burgers = doc1[5]
print(french_fries, "<->", burgers, french_fries.similarity(burgers))